# Employee Database Manager
A Python and PostgreSQL learning project: create, read, update and delete employee records, filter salaries and produce a salary summary.

## Before running
Start PostgreSQL and create `company_db` in pgAdmin if it does not exist. Use this notebook with your practice database. Run cells from top to bottom and enter your database password when prompted. The password is not stored in the notebook.

The setup inserts missing sample IDs and leaves existing records unchanged. The optional update/delete examples are commented out to avoid changing records during Run All.

In [ ]:
%pip install psycopg2-binary

## 1. Connect
`conn` is the database connection. `cursor` sends SQL and retrieves results.

In [ ]:
import psycopg2
from getpass import getpass

conn = psycopg2.connect(
    host="localhost",
    database="company_db",
    user="postgres",
    password=getpass("Enter your database password: "),
    port=5432
)
cursor = conn.cursor()
print("Connected successfully")

## 2. Create the table and load sample records
The primary key prevents duplicate IDs. `ON CONFLICT (id) DO NOTHING` skips IDs already present. `commit()` saves the transaction; `rollback()` undoes uncommitted work if SQL fails. An existing table is not altered by this setup.

In [ ]:
sample_employees = [
    (1, "John", 50000),
    (2, "Sarah", 60000),
    (3, "David", 55000),
    (4, "Emma", 65000),
    (5, "Alex", 58000),
    (6, "Omar", 68000),
]

try:
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            salary NUMERIC(12, 2) NOT NULL CHECK (salary >= 0)
        );
    """)
    cursor.executemany("""
        INSERT INTO employees (id, name, salary)
        VALUES (%s, %s, %s)
        ON CONFLICT (id) DO NOTHING;
    """, sample_employees)
    conn.commit()
    print("Sample data ready.")
except Exception:
    conn.rollback()
    raise

## 3. Read and filter employees
`%s` placeholders keep values separate from SQL. `(min_salary,)` is a one-item tuple. `fetchall()` retrieves the result rows.

In [ ]:
def list_employees():
    try:
        cursor.execute("SELECT id, name, salary FROM employees ORDER BY id;")
        rows = cursor.fetchall()
        conn.commit()
        return rows
    except Exception:
        conn.rollback()
        raise


def get_employees_above(min_salary):
    try:
        cursor.execute(
            "SELECT id, name, salary FROM employees WHERE salary > %s ORDER BY id;",
            (min_salary,)
        )
        rows = cursor.fetchall()
        conn.commit()
        return rows
    except Exception:
        conn.rollback()
        raise

## 4. Add, update and delete
`RETURNING` lets PostgreSQL return the affected record. `fetchone()` retrieves it, or returns `None` if no row was affected. Each function saves successful work and rolls back on failure.

In [ ]:
def add_employee(employee_id, name, salary):
    try:
        cursor.execute(
            """
            INSERT INTO employees(id, name, salary)
            VALUES (%s, %s, %s)
            ON CONFLICT (id) DO NOTHING
            RETURNING id;
            """,
            (employee_id, name, salary)
        )
        added = cursor.fetchone()
        conn.commit()

        if added:
            print ("Employee added.")
        else:
            print("That employee already exists.")

    except Exception:
        conn.rollback()
        raise

In [ ]:
def update_salary(employee_id, new_salary):
    try:
        cursor.execute(
            """
            UPDATE employees
            SET salary = %s
            WHERE id = %s
            RETURNING id, name, salary;
            """,
            (new_salary, employee_id)
        )
        updated = cursor.fetchone()
        conn.commit()

        if updated:
            print ("Updated:", updated)
        else:
            print("Employee ID not found.")

    except Exception:
        conn.rollback()
        raise

In [ ]:
def delete_employee(employee_id):
    try:
        cursor.execute(
            """
            DELETE FROM employees
            WHERE id = %s
            RETURNING id, name;
            """,
            (employee_id,)
        )
        deleted = cursor.fetchone()
        conn.commit()

        if deleted:
            print("Deleted:", deleted)
        else:
            print("Employee Id not found.")

    except Exception:
        conn.rollback()
        raise

## 5. Salary summary
`COUNT`, `SUM`, `AVG`, `MIN` and `MAX` summarize the table. For an empty table, count is zero and the other aggregates are `None`.

In [ ]:
def salary_summary():
    try:
        cursor.execute("""
            SELECT COUNT(*), SUM(salary),
                   ROUND(AVG(salary)::numeric, 2),
                   MIN(salary), MAX(salary)
            FROM employees;
        """)
        count, total, average, lowest, highest = cursor.fetchone()
        conn.commit()
    except Exception:
        conn.rollback()
        raise

    print("Employees:", count)
    print("Total salaries:", total)
    print("Average salary:", average)
    print("Lowest salary:", lowest)
    print("Highest salary:", highest)

## 6. Run the report
With exactly the six sample records: total salary = 356000, average = 59333.33, minimum = 50000 and maximum = 68000. Existing edits or additional records can change these results.

In [ ]:
for row in list_employees():
    print(row)

print("\nEmployees earning more than 60000:")
for row in get_employees_above(60000):
    print(row)

print("\nSalary summary:")
salary_summary()

## 7. Optional practice
Uncomment the calls you want to try while the connection is open. Use an unused ID for a test employee. Delete only a practice record you intend to remove.

In [ ]:
# add_employee(99, "Test Employee", 4000)
# update_salary(99, 4500)
# delete_employee(99)
# salary_summary()

## 8. Close the connection
Run this when finished. To use the functions again, rerun the connection cell first.

In [ ]:
cursor.close()
conn.close()
print("Connection closed.")